# HW2 — Linear Models + Regularization, Validated in Time

**FIN 7057 · Due after Week 5 · 100 pts**

Fill in every `# TODO`. Runs top-to-bottom. See `README.md` for the rubric. Follow
the course discipline: past-only features, strictly ordered train/validation/test periods,
preprocessing fitted inside each time-series cross-validation fold, and a test period used once.
Part C compares shuffled and walk-forward validation; the comparison's numerical direction is an
empirical result, while the invalidity of shuffling time-dependent observations is a design fact.


## 0. Setup & reproducibility


In [ ]:
# !pip install -q yfinance scikit-learn scipy
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.stats import spearmanr
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet, LogisticRegression
from sklearn.model_selection import TimeSeriesSplit, KFold, GridSearchCV, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.metrics import r2_score, roc_auc_score, log_loss, brier_score_loss
SEED=42; np.random.seed(SEED)
TICKER, START, END = 'SPY', '2010-01-01', '2024-12-31'


In [ ]:
# Provided loader (synthetic fallback) — reuse your HW1 source if you prefer.
def load_prices(ticker, start, end):
    try:
        import yfinance as yf
        df=yf.download(ticker,start=start,end=end,progress=False,auto_adjust=True)
        if len(df): s=df['Close'].squeeze(); s.name='close'; s.index.name='date'; return s.to_frame()
        raise RuntimeError('empty')
    except Exception as e:
        print(f'synthetic ({e!r})'); n=252*15; dates=pd.bdate_range(start=start,periods=n)
        rng=np.random.default_rng(SEED); r=rng.standard_normal(n)*0.01
        return pd.DataFrame({'close':100*np.exp(np.cumsum(r))}, index=pd.Index(dates,name='date'))
px=load_prices(TICKER,START,END); px['ret']=np.log(px['close']).diff(); px=px.dropna()

def spearman_corr(pred, actual):
    pred=np.asarray(pred,float); actual=np.asarray(actual,float)
    if pred.std()==0 or actual.std()==0: return float('nan')   # constant -> undefined
    return spearmanr(pred, actual).statistic


## Part A — Features (>=8, leak-free) + three-way time split


In [ ]:
def build_features(prices):
    p=prices.copy(); p['ret']=np.log(p['close']).diff(); f=pd.DataFrame(index=p.index)
    f['ret_lag1']=p['ret'].shift(1)
    for w in (5,21,63): f[f'mom{w}']=p['ret'].rolling(w).mean().shift(1)
    f['vol21']=p['ret'].rolling(21).std().shift(1)
    # TODO: add features to reach >=8 total (e.g., more lags/windows, rolling z-score, RSI,
    #       a couple you suspect are weak). Keep them PAST-ONLY (shift!).
    return f



def assert_feature_future_invariance(make_features, prices, t0_frac=0.6):
    """Provided diagnostic: later prices must not alter earlier feature values."""
    base = make_features(prices)
    perturbed = prices.copy()
    t0 = int(len(prices) * t0_frac)
    perturbed.iloc[t0 + 1:, perturbed.columns.get_loc('close')] *= 1.25
    changed = make_features(perturbed)
    delta = (base.iloc[:t0 + 1] - changed.iloc[:t0 + 1]).abs().to_numpy()
    delta = delta[np.isfinite(delta)]
    return float(delta.max()) if delta.size else 0.0

feature_disc = assert_feature_future_invariance(build_features, px)
print(f'Max past-feature change after perturbing the future: {feature_disc:.2e}')

feat=build_features(px)
target=px['ret'].shift(-1)                 # next-day return (regression)
data=feat.join(target.rename('y')).dropna()

# TODO: three-way split BY DATE: train (~60%) / val (~20%) / test (~20%). State the dates.
n=len(data); i1,i2=int(n*0.6), int(n*0.8)
train, val, test = data.iloc[:i1], data.iloc[i1:i2], data.iloc[i2:]
print('train', train.index.min().date(),'->',train.index.max().date())
print('val  ', val.index.min().date(),'->',val.index.max().date())
print('test ', test.index.min().date(),'->',test.index.max().date())
cols=list(feat.columns)


## Part B — Scale on train only; fit four models (tune alpha with TimeSeriesSplit)


In [ ]:
# Keep raw design matrices here. Each tuned estimator is a Pipeline, so its scaler is
# fitted again inside every TimeSeriesSplit training fold.
Xtr, Xva, Xte = (df[cols].to_numpy() for df in (train, val, test))
ytr, yva, yte = train['y'].values, val['y'].values, test['y'].values

tscv = TimeSeriesSplit(5)
alphas = np.logspace(-4, 1, 30)
models = {
    'OLS': make_pipeline(StandardScaler(), LinearRegression()),
    'Ridge': GridSearchCV(
        make_pipeline(StandardScaler(), Ridge()),
        {'ridge__alpha': alphas}, cv=tscv, scoring='neg_mean_squared_error'),
    'Lasso': GridSearchCV(
        make_pipeline(StandardScaler(), Lasso(max_iter=20000, random_state=SEED)),
        {'lasso__alpha': alphas}, cv=tscv, scoring='neg_mean_squared_error'),
    'ElasticNet': GridSearchCV(
        make_pipeline(StandardScaler(), ElasticNet(max_iter=20000, random_state=SEED)),
        {'elasticnet__alpha': alphas, 'elasticnet__l1_ratio': [.2, .5, .8]},
        cv=tscv, scoring='neg_mean_squared_error'),
}
for name, model in models.items():
    model.fit(Xtr, ytr)
print('fitted:', list(models))
for name in ('Ridge', 'Lasso', 'ElasticNet'):
    print(f'{name:10s} {models[name].best_params_}')


## Part C — Shuffled k-fold vs walk-forward (the point)
Estimate Ridge performance two ways on the TRAIN data and compare. Try an overlapping label
(below) to make the gap vivid, then explain which you trust.


In [ ]:
# Overlapping forward label to expose leakage under shuffling:
fwd = px['ret'].shift(-1).rolling(10).mean().shift(-9)
dd  = feat.join(fwd.rename('y')).dropna()
Xc  = dd[cols].to_numpy(); yc = dd['y'].values
ridge = Ridge(alpha=1.0)
kf = cross_val_score(ridge, Xc, yc, cv=KFold(5, shuffle=True, random_state=SEED), scoring='r2').mean()
ts = cross_val_score(ridge, Xc, yc, cv=TimeSeriesSplit(5), scoring='r2').mean()
print(f'Shuffled KFold CV R2:  {kf:+.4f}')
print(f'TimeSeriesSplit CV R2: {ts:+.4f}')
# TODO (written): which is optimistic? which do you trust? why? (Part C2/C3)


## Part D — In-sample vs out-of-sample table


In [ ]:
rows=[]
for name,m in models.items():
    rows.append({
        'model':name,
        'train_R2':  r2_score(ytr, m.predict(Xtr)),
        'test_R2':   r2_score(yte, m.predict(Xte)),
        'train_Spearman':  spearman_corr(m.predict(Xtr), ytr),
        'test_Spearman':   spearman_corr(m.predict(Xte), yte),
    })
tbl=pd.DataFrame(rows).set_index('model').round(4); print(tbl)
base_r2 = r2_score(yte, np.full_like(yte, ytr.mean()))
print(f'\nBaseline (predict train mean) test R2: {base_r2:+.4f}')
# TODO (written, >=150 words): which model overfit most? did regularization help OOS? beat baseline?


## Part E — Feature selection, stability, reflection


In [ ]:
lasso = models['Lasso'].best_estimator_.named_steps['lasso']
kept = [c for c,coef in zip(cols, lasso.coef_) if coef!=0]
print(f'Lasso kept {len(kept)}/{len(cols)} features:', kept)

# Coefficient stability: refit Ridge on two halves of TRAIN and compare signs.
h=len(Xtr)//2
a=make_pipeline(StandardScaler(), Ridge(alpha=1.0)).fit(Xtr[:h],ytr[:h]).named_steps['ridge'].coef_
b=make_pipeline(StandardScaler(), Ridge(alpha=1.0)).fit(Xtr[h:],ytr[h:]).named_steps['ridge'].coef_
stab=pd.DataFrame({'feature':cols,'first_half':a.round(3),'second_half':b.round(3)})
print(stab.to_string(index=False))
# TODO (written, >=150 words): is test Spearman correlation meaningfully != 0? what next? what is OFF-LIMITS for the test set?


## Part F — The same information, as a decision

Everything above forecast a *magnitude*. Now ask the same features for a *decision*, on the same
split. Three separate contests: **ranking** (AUROC), **probability quality** (calibration, Brier,
log loss), and **the value of one policy** (a threshold chosen from declared costs).

Direction on daily returns is close to a coin flip. An AUROC near 0.5 with no gain over the base
rate is the expected result and earns full marks — keeping the three contests apart is what is
graded.

In [ ]:
# Part F.1 — the same features, a direction target, the same split.
ytr_c = (ytr > 0).astype(int)
yva_c = (yva > 0).astype(int)
yte_c = (yte > 0).astype(int)
base_rate = ytr_c.mean()                       # the feasible zero-skill forecast
print(f"training up-day rate (the baseline forecast): {base_rate:.4f}")

clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
clf.fit(Xtr, ytr_c)
p_va = clf.predict_proba(Xva)[:, 1]
p_te = clf.predict_proba(Xte)[:, 1]

# TODO: report test AUROC and log loss, each beside the base-rate forecast.
#   roc_auc_score(yte_c, p_te)              vs 0.5
#   log_loss(yte_c, p_te)                   vs log_loss(yte_c, np.full_like(p_te, base_rate))

# Part F.2 — is it calibrated? Reliability curve on VALIDATION, with bin counts.
# TODO: bin p_va (e.g. 10 bins), plot observed frequency against mean predicted
#       probability, annotate each bin's count, and report Brier vs the base rate.

# Part F.3 — choose the threshold from costs, on validation, then freeze it.
C_FP, C_FN = 1.0, 1.0        # TODO: declare your costs and justify the ratio in one sentence
def expected_cost(y_true, p, thr, c_fp=C_FP, c_fn=C_FN):
    flag = p >= thr
    fp = int(((flag == 1) & (y_true == 0)).sum())
    fn = int(((flag == 0) & (y_true == 1)).sum())
    return (c_fp * fp + c_fn * fn) / len(y_true)

grid = np.linspace(0.05, 0.95, 91)
# TODO: pick the validation-minimizing threshold, freeze it, then report action rate
#       and realized cost on TEST beside a 0.50 threshold. Do not tune on test.
thr_star = None


## Extensions (optional)

The required core is everything above, and the self-check tests only that. These extensions are
optional and not required for full marks:

- Plot Ridge, Lasso, and ElasticNet coefficient paths over the regularization grid.
- Measure feature/coefficient stability across all time-series folds, not just two halves.
- Prespecify a second prediction horizon and evaluate it on a separate forward window.


## Self-check (auto-graded)

Run this cell **last**, after the whole notebook. Every line must print **PASS** before you submit — it verifies the mechanical requirements (reproducibility, leak-free features, time-ordered splits, metrics computed). Your written answers and judgment are graded by hand.

In [ ]:
# ================= SELF-CHECK (auto-graded) =================
# Run LAST. Each line reports PASS/FAIL for a mechanical requirement.
# Written answers / judgment are graded separately, by hand.
def _check(name, fn, hint=""):
    try:
        ok = bool(fn())
    except Exception as e:
        ok, hint = False, hint or f"({type(e).__name__}: {e})"
    print(("PASS  " if ok else "FAIL  ") + name + ("" if ok else "   -> " + hint))
    return ok
_r = []
_r.append(_check("reproducibility: SEED set", lambda: SEED is not None, "set SEED"))
_r.append(_check(">= 8 features (feat)", lambda: feat.shape[1] >= 8, "reach >= 8 leak-free features in build_features()"))
_r.append(_check("future perturbation leaves past features unchanged", lambda: feature_disc < 1e-8, "remove centered/forward windows and full-sample feature statistics"))
_r.append(_check("three-way split time-ordered & disjoint", lambda: train.index.max() < val.index.min() and val.index.max() < test.index.min(), "split by date into train < val < test, no overlap"))
_r.append(_check("preprocessing lives inside tuned CV pipelines", lambda: all(hasattr(models[n], 'best_estimator_') and 'standardscaler' in models[n].best_estimator_.named_steps for n in ('Ridge','Lasso','ElasticNet')), "put StandardScaler in each GridSearchCV pipeline"))
_r.append(_check("modelling frame has no NaNs", lambda: data[cols + ['y']].notna().all().all(), "dropna() so features and target y are complete"))
_r.append(_check("single-series Spearman metric works", lambda: callable(spearman_corr) and np.isfinite(spearman_corr(np.arange(len(test)), test['y'].values)), "keep the provided spearman_corr()"))
_r.append(_check("Part F: classifier fitted on the direction target", lambda: set(np.unique(ytr_c)) <= {0, 1} and hasattr(clf, "predict_proba"), "fit a logistic model on (y > 0) with the same features"))
_r.append(_check("Part F: threshold selected on validation, not test", lambda: thr_star is not None and abs(thr_star - grid[int(np.argmin([expected_cost(yva_c, p_va, t) for t in grid]))]) < 1e-9, "pick thr_star by minimizing expected cost on the VALIDATION block, then freeze it"))
_r.append(_check("Part F: both error costs declared and positive", lambda: (C_FP > 0 and C_FN > 0), "declare C_FP and C_FN and justify the ratio"))
print(f"\n{sum(_r)}/{len(_r)} mechanical checks passed.")
assert all(_r), "Fix the FAIL items above before submitting."


## AI-use disclosure (required)

State specifically whether and how you used AI tools. You remain responsible for every claim,
decision, line of code, citation, and interpretation, and you must be able to explain them.

## Reproducibility checklist
- [ ] Restart & Run All  - [ ] seed  - [ ] dates  - [ ] data source  - [ ] packages
